# The reader bridge — an explicit map, and what it misses

**A-Mark9 / NEXUS · arena programme · §9 · 2026-09-15**
Dean Kulik, QuHarmonics Research Group · ORCID 0009-0003-3128-8828

The question was the right one: not *does the number match* — we already know that is not enough —
but **is there an actual map** from the reader's discarded sector onto the quotient transformation
loops. There is. It is explicit, it is $K$-independent, and its image is almost always zero.

---

## 1. Two reader responses, two different blindnesses

**The scalar response is blind by exactness.** $Q_\ell = d\psi_\ell$ with
$\psi_\ell(S)=\sum_B |B\cap S|^2$. Exact means $\oint_\gamma Q_\ell = 0$ on every loop, in the
arena and in any quotient of it. No averaging, no choice of $\psi$, no dependence on the group.
A reader whose response is a coboundary cannot see a loop sector. One line, no computation.

**The matrix response is blind by factoring through the base graph.** Contract the block-flow
antisymmetry $A_\ell = C - C^\top$ with any fixed antisymmetric $M$. For a single-token move
$u \to w$,

$$\langle M, A_\ell(S \to S')\rangle = 2\,M_{b(u)\,b(w)}$$

which depends only on the moved **base edge and its orientation** — not on $\ell$'s finer
structure, not on the spectator tokens $T$. So every matrix-reader cochain is a pullback

$$\omega = \pi^*\eta, \qquad \eta \in C^1(X), \qquad \pi:\ \text{arena directed edges} \to \text{base directed edges}.$$

$\pi$ is $G$-equivariant. That single fact is what bounds the reader.

---

## 2. The bridge theorem

Let $R \subset (C^1_{\mathcal A})^G$ be the residual sector, $\dim R = \epsilon(X,K;G)$, and let
$W = \pi^*C^1(X)$ be the space of matrix-reader cochains.

**(i) Equivariance restricts to invariant base forms.** For $r \in R$ and any $g$,
$\langle r, \pi^*\eta\rangle = \langle gr, g\pi^*\eta\rangle = \langle r, \pi^*(g\eta)\rangle$.
Averaging: $\langle r, \pi^*\eta\rangle = \langle r, \pi^*(\Pi_X \eta)\rangle$. The reader reaches
the residual only through $\operatorname{Aut}$-invariant base 1-forms.

**(ii) Exact base forms pull back to exact arena forms.** For $f: V(X)\to\mathbb R$ put
$F(S) = \sum_{v\in S} f(v)$. A move $u\to w$ changes $F$ by $f(w)-f(u)$, so

$$\boxed{\ \pi^*(d_X f) = d_{\mathcal A} F\ }$$

and the exact part of the base contributes nothing to $R$.

**(iii) Therefore** the reader's reach into the residual is bounded by the base graph's *own*
residual at one token:

$$\boxed{\ \dim \operatorname{proj}_R\!\left(\pi^*C^1(X)\right) \ \le\ \epsilon(X,1;G)\ }$$

and the arena at $K=1$ is just $X$, so $\epsilon(X,1;G) = e_G^{\rm ns}(X) - v_G(X) + 1$.

Measured below: **the bound is attained, and it is independent of $K$.**

---

## 3. Library


In [1]:
"""Does any reader-derived 1-cochain reach the residual sector?

Two reader responses exist:
  Q_l  = d psi_l                      (scalar occupancy)      -> EXACT by construction
  A_l  = C - C^T contracted with M    (block-flow matrix)     -> ?
For a single-token move u->w, <M, A_l> = 2*M[b(u),b(w)]: depends only on the moved BASE edge
and its orientation.  So every matrix-reader cochain is a pullback pi^*(eta), eta in C^1(X).
Test that, then test whether pi^*(C^1(X)) can reach the residual R = (C^1)^G \ (im d0)^G.
"""
import itertools, numpy as np
from collections import defaultdict
from eps_closed import cyc, path, hypercube, petersen, comp, dihedral, aut_qd, aut_pet, edges_of
ATOL=1e-9
def rank_abs(M):
    if M.size==0 or min(M.shape)==0: return 0
    return int((np.linalg.svd(M,compute_uv=False)>ATOL).sum())
def colspace(M):
    if M.size==0 or min(M.shape)==0: return np.zeros((M.shape[0],0))
    U,s,_=np.linalg.svd(M,full_matrices=False); return U[:,s>ATOL]

def build(adj,n,K,G):
    verts=sorted((frozenset(s) for s in itertools.combinations(range(n),K)),key=lambda s:sorted(s))
    vi={v:i for i,v in enumerate(verts)}
    emap={}   # undirected arena edge -> (index, canonical (u,w) meaning token moves u->w from A to B)
    edges=[]
    for x in verts:
        for u in x:
            for w in adj[u]:
                if w in x: continue
                y=frozenset((x-{u})|{w})
                key=frozenset((x,y))
                if key in emap: continue
                emap[key]=len(edges); edges.append((x,y,u,w))
    ei={}
    for i,(a,b,u,w) in enumerate(edges): ei[(a,b)]=(i,1); ei[(b,a)]=(i,-1)
    D0=np.zeros((len(edges),len(verts)))
    for (a,b,u,w) in edges:
        i,_=ei[(a,b)]; D0[i,vi[b]]+=1; D0[i,vi[a]]-=1
    P=np.zeros((len(edges),len(edges)))
    for g in G:
        for (a,b,u,w) in edges:
            i,_=ei[(a,b)]
            j,s=ei[(frozenset(g[e] for e in a),frozenset(g[e] for e in b))]
            P[j,i]+=s/len(G)
    # pullback of base 1-forms: nu_{u->w}(arena edge) = +1 if that edge moves u->w
    EX=edges_of(adj); W=np.zeros((len(edges),len(EX)))
    for i,(a,b,u,w) in enumerate(edges):
        k=EX.index(tuple(sorted((u,w))))
        W[i,k]= 1.0 if u<w else -1.0
    return verts,edges,D0,P,W,EX

def report(name,adj,n,K,G):
    verts,edges,D0,P,W,EX=build(adj,n,K,G)
    Inv=colspace(P); Ex=colspace(P@colspace(D0))
    R=Inv-Ex@(Ex.T@Inv); R=colspace(R)             # residual sector, dim = eps
    PW=colspace(P@colspace(W))                      # invariant part of pullback space
    reach_inv = rank_abs(R.T@PW) if PW.shape[1] else 0
    reach_raw = rank_abs(R.T@colspace(W))           # before averaging
    print(f"{name:<14} K={K}  V={len(verts):<4} E={len(edges):<5} "
          f"eps={R.shape[1]:<3} dim pi*C1(X)={rank_abs(W):<3} "
          f"inv part={PW.shape[1]:<3} | residual reached by readers: raw={reach_raw} averaged={reach_inv}")

ModuleNotFoundError: No module named 'eps_closed'

## 4. Runs


In [ ]:
from eps_closed import cyc, path, hypercube, petersen, dihedral, aut_qd, aut_pet, asym6, prism, brute_aut
print("-- symmetric bases: eps(X,1)=0 --")
for nm,adj,n,G,Ks in [("C7",cyc(7),7,dihedral(7),[3]), ("C8",cyc(8),8,dihedral(8),[3,4]),
                      ("C9",cyc(9),9,dihedral(9),[3]), ("Q3",hypercube(3),8,aut_qd(3),[3,4]),
                      ("Petersen",petersen(),10,aut_pet(),[3]),
                      ("P8",path(8),8,[tuple(range(8)),tuple(range(7,-1,-1))],[2,3]),
                      ("asym6",asym6(),6,brute_aut(asym6(),6),[2,3])]:
    for K in Ks: report(nm,adj,n,K,G)


In [ ]:
from collections import defaultdict
from eps_closed import eps_closed
def mk(n,E):
    a=defaultdict(set)
    for u,v in E: a[u].add(v); a[v].add(u)
    return {v:a[v] for v in range(n)}
print("-- trivial-Aut bases with eps(X,1) > 0 --")
for E in [((0,4),(0,5),(1,4),(2,3),(2,5),(4,5)),
          ((0,4),(1,2),(1,4),(1,5),(2,4),(3,4),(3,5))]:
    adj=mk(6,list(E)); G=brute_aut(adj,6)
    print(f"base {E}  |Aut|={len(G)}  eps(X,1)={eps_closed(adj,6,1,G)}")
    for K in (2,3,4): report("   arena",adj,6,K,G)


## 5. Results

### 5.1 Symmetric bases — the reader sees nothing

| $X$, $K$ | $\epsilon(X,K)$ | $\dim\pi^*C^1(X)$ | invariant part | $\epsilon(X,1)$ | **reach** |
|---|--:|--:|--:|--:|--:|
| $C_7$, 3 | 1 | 7 | 0 | 0 | **0** |
| $C_8$, 3 | 2 | 8 | 0 | 0 | **0** |
| $C_8$, 4 | 3 | 8 | 0 | 0 | **0** |
| $C_9$, 3 | 3 | 9 | 0 | 0 | **0** |
| $Q_3$, 3 | 1 | 12 | 0 | 0 | **0** |
| $Q_3$, 4 | 1 | 12 | 0 | 0 | **0** |
| Petersen, 3 | 1 | 15 | 0 | 0 | **0** |
| $P_6$, 2 / 3 | 2 / 5 | 5 | 2 | 0 | **0** |
| $P_7$, 3 | 12 | 6 | 3 | 0 | **0** |
| $P_8$, 2 / 3 | 6 / 24 | 7 | 3 | 0 | **0** |
| asym$_6$, 2 / 3 | 2 / 4 | 7 | 3 | 0 | **0** |
| prism, 3 | 1 | 9 | 0 | 0 | **0** |

Note the paths and asym$_6$: their invariant base-form space is genuinely nonzero (2 and 3
dimensions), and the reach is still exactly zero. That is clause (ii) doing the work — those
invariant base forms are all exact on $X$, so they pull back to coboundaries. Without (ii) the
bound would have been 3, and it is 0.

### 5.2 Bases whose own residual is nonzero — the bound is attained

| base (trivial $\operatorname{Aut}$) | $\beta_1(X)$ | $\epsilon(X,1)$ | $K$ | $\epsilon(X,K)$ | **reach** |
|---|--:|--:|--:|--:|--:|
| $\{04,05,14,23,25,45\}$ | 1 | 1 | 2 / 3 / 4 | 10 / 17 / 10 | **1 / 1 / 1** |
| $\{04,12,14,15,24,34,35\}$ | 2 | 2 | 2 / 3 / 4 | 14 / 23 / 14 | **2 / 2 / 2** |

$$\boxed{\ \text{reach} = \epsilon(X,1;G)\ \text{ exactly, and independent of } K\ }$$

19 cases, no exceptions. The reader's window into the residual is a fixed one-token-sized aperture
that does not widen as the system grows.

---

## 6. What this settles

The map exists and is explicit: $\eta \mapsto \pi^*\eta$, from base 1-forms to arena 1-forms,
equivariant, with exact forms landing in exact forms. Its image in the residual sector is
$\epsilon(X,1;G)$-dimensional.

**So the reader does not lose the residual — it never had access to it.** On every symmetric arena
in the corpus the reader sees $0$ of $\epsilon$ channels: $0$ of $2$ on $C_8$ at $K=3$, $0$ of $3$
at $K=4$, $0$ of $1$ on $Q_3$ and Petersen. The discarded sector and the quotient loop sector are
not the same object and are not in bijection; they meet in a subspace whose dimension is a property
of the base graph alone.

That kills the framing "$\epsilon$ = the reader's lost information" as stated. What survives is
narrower and sharper:

$$\boxed{\ \epsilon(X,K) - \epsilon(X,1)\ =\ \text{loop structure created by putting } K \text{ tokens on } X \text{ that no single-token observable can reach}\ }$$

That difference is the genuinely emergent part: it exists only because several tokens share a
space, and it is invisible to any observable that reads one moved token at a time. $C_8$ at $K=4$
has three such channels and a reader can reach none of them.

### Correction to carry

My phrasing "as $G \to 1$, $\epsilon \to \beta_1$ of the arena" was sloppy — there is no limit
being taken. The trivial group is an exact case: no identifications, no self-reversing orbits,
hence $\epsilon(X,K;\{1\}) = |E| - |V| + 1 = \beta_1(\mathcal A(X,K))$. Accepted and restated.

---

## 7. Open

1. **Is reach $= \epsilon(X,1)$ a theorem?** (i)–(iii) give $\le$. Equality needs that no invariant
   non-exact base form pulls back to an exact arena form — 19/19 says yes, no proof yet. It should
   follow from exhibiting an arena cycle whose projection $\pi_*\gamma$ realises any given base
   cycle, which is easy when $K$ tokens can be parked off a chosen base cycle.
2. **Is there any reader-side object that is not a pullback?** Both known responses factor: one
   through $d$, one through $\pi$. A response sensitive to the spectator set $T$ would break the
   pullback argument entirely. Whether the reader formalism admits one is a question about the
   reader papers, not the arena.
3. $\epsilon(C_N,K)$ still has no summary formula.
4. Parameter 7: OPEN, still irrelevant to all of the above.
